## 뉴스 가져오기

In [ ]:
from wrapper.news_fetcher import NewsFetcher
from wrapper.llm_wrapper import LLM
from wrapper.api_wrapper import ApiWrapper
from tqdm import tqdm

from entity.entity import *


news = NewsFetcher()

news_list = news.fetch_news(n_pages=3)
news.save_csv()

news_tags = [news.tags[i][0] for i in news.tags.keys()]
news_ids = {tag: [] for tag in news_tags}

for tag in news_tags:
    news.news[tag].pop(0)

#### 뉴스 데이터 가져오기

In [ ]:
from wrapper.news_fetcher import NewsFetcher


news = NewsFetcher().load_csv()

## 뉴스 업로드

In [ ]:
from wrapper.api_wrapper import ApiWrapper


api = ApiWrapper()
uploaded_news = api.upload_news(news)

#### failback

In [ ]:
from wrapper.api_wrapper import ApiWrapper
from entity.entity import *


api = ApiWrapper()
on_server = api.download_news()
uploaded_news: list[UploadedNews] = []

for tag in news:
    for n in news[tag]:
        for o in on_server:
            if o["title"] == n.title:
                uploaded_news.append(
                    UploadedNews(
                        title=n.title,
                        content=n.content,
                        image=n.image,
                        press=n.press,
                        pub_time=n.pub_time,
                        tag=n.tag,
                        url=n.url,
                        id=o["newsIdx"],
                    )
                )
                break

len(uploaded_news)

api.save_csv(uploaded_news)

In [ ]:
from wrapper.api_wrapper import ApiWrapper
from entity.entity import *


api = ApiWrapper()
uploaded_news = api.load_csv()

# 뉴스 요약

#### 뉴스 요약 진행

In [ ]:
from wrapper.llm_wrapper import LLM
from wrapper.api_wrapper import ApiWrapper
from tqdm import tqdm

from entity.entity import *


api = ApiWrapper()


llm = LLM(n_ctx=8192, max_tokens=1024)
summerized_news: list[SummerizedNews] = []

for news in tqdm(uploaded_news):
    llm.set_prompt(
        f"""
        [요청 사항]
        - 이 뉴스를 다음 양식을 준수하는 세 문장으로 요약해 주세요.

        [준수 사항]
        - 첫 번째 문장은 이 기사에서 다루는 핵심 사건을 설명하는 80자 내외의 완결된 문장이어야 합니다.
        - 두 번째 문장은 사건의 배경과 관련된 맥락을 설명하는 80자 내외의 완결된 문장이어야 합니다.
        - 세 번째 문장은 사건의 진행과 결과를 설명하는 80자 내외의 완결된 문장이어야 합니다.

        [참고 사항]
        - 요약하신 자료는 텍스트 임베딩을 거쳐 클러스터링 작업에 사용될 것입니다.
        - 이 뉴스는 {news.tag} 분야의 뉴스입니다.

        [예시]
        1. 지난 23일 16시 경 광주 광산구 아파트 주차장에서 차량 4대를 들이받고 벤츠를 버린 운전자가 사건 발생 12시간 만에 경찰에 자진 출석했다.
        2. 사고 직후 운전자는 아무런 조치 없이 연락처와 벤츠를 남기고 도주했으며, 사고 12시간 40분 만에 같은 날 오후 6시쯤 경찰에 출석했다.
        3. 경찰은 A씨를 들이받은 차량을 수습하지 않은 채 도주한 혐의를 적용하여 조사하고 있으며, CCTV 등을 통해 운전 경로를 추적 수사할 방침이다.

        """
    )

    content = llm.generate(
        instruction=
        f"""
        [뉴스 제목]
        {news.title}
        [뉴스 내용]
        {news.content}
        """[:8191],
        reset_prompt=True
    )

    summerized_news.append(SummerizedNews(title=news.title, content=content, topics="", id=news.id))


In [ ]:
for i in summerized_news:
    print(i.content, end="\n\n")

In [ ]:
import pickle

with open("summerized.pkl", "wb") as f:
    pickle.dump(summerized_news, f)


In [ ]:
import pickle


with open("summerized.pkl", "rb") as f:
    summerized_news = pickle.load(f)


In [ ]:
from entity.entity import *


def get_summerized_news(id: int) -> SummerizedNews:
    for i in summerized_news:
        if i.id == id:
            return i
    
    return None

# 클러스터링

### 요약문 임베딩

In [ ]:
# from wrapper.llm_wrapper import LLM
# model = LLM(embedding=True).model
from llama_cpp import Llama

MODEL_PATH = "/home/gpp/src/model/llama3-korean-bllossom-8b/llama-3-Korean-Bllossom-8B-Q4_K_M.gguf"
model = Llama(model_path=MODEL_PATH, embedding=True, verbose=False)
clusters = {}

In [ ]:
import numpy as np
from tqdm import tqdm

firsts = []
seconds = []
thirds = []

for texts in tqdm(summerized_news):
    first, second, third = list(filter(lambda x: x.strip() != '', texts.content.split('\n')))
    firsts.append(np.mean(model.embed(first, normalize=True), axis=0))
    seconds.append(np.mean(model.embed(second, normalize=True), axis=0))
    thirds.append(np.mean(model.embed(third, normalize=True), axis=0))

assert len(firsts) == len(seconds) == len(thirds)

first_embeddings = np.array([np.pad(embedding, (0, max(len(e) for e in firsts) - len(embedding)), 'constant') for embedding in firsts])
second_embeddings = np.array([np.pad(embedding, (0, max(len(e) for e in seconds) - len(embedding)), 'constant') for embedding in seconds])
third_embeddings = np.array([np.pad(embedding, (0, max(len(e) for e in thirds) - len(embedding)), 'constant') for embedding in thirds])



#### OPTION 1: cosine similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity


# 각 문장별 유사도 행렬 계산
first_similarity = cosine_similarity(first_embeddings)
second_similarity = cosine_similarity(second_embeddings)
third_similarity = cosine_similarity(third_embeddings)

# 세 유사도의 평균값을 최종 유사도로 사용
similarity_matrix = (first_similarity + second_similarity + third_similarity) / 3
metric="precomputed"

#### OPTION 2: pairwise distance(euclidian)

In [ ]:
from sklearn.metrics import pairwise_distances


first_similarity = pairwise_distances(first_embeddings)
second_similarity = pairwise_distances(second_embeddings)
third_similarity = pairwise_distances(third_embeddings)

# 세 유사도의 평균값을 최종 유사도로 사용
similarity_matrix = (first_similarity + second_similarity + third_similarity) / 3
metric="euclidean"

#### cluster and show result

In [ ]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=0.4, min_samples=3, metric=metric)
clusters_ = dbscan.fit_predict(similarity_matrix)

# 클러스터 결과
print(len(set(clusters_)))
clusters = {}  # 클러스터를 저장할 딕셔너리

for cluster_id in set(clusters_):
    cluster_news_ids = set()  # 중복을 피하기 위해 set 사용

    if cluster_id != -1:  # -1은 노이즈
        print(f"Cluster {cluster_id}:")
        for i in np.where(clusters_ == cluster_id)[0]:
            # 중복된 ID를 추가하지 않도록 set에 추가
            cluster_news_ids.add(summerized_news[i].id)
            print(summerized_news[i].id, summerized_news[i].title)

        clusters[cluster_id] = list(cluster_news_ids)


#### (수작업) 가장 잘 군집화 된 군집 선택해서 뉴스 제작 진행

In [ ]:
from wrapper.llm_wrapper import LLM


# del model
# del llm
llm = LLM(max_tokens=8192)#, temperature=0.6, top_p=0.7)

top_clusters = []
top_clusters.append(np.argmax(clusters))

def news_creation_chain(llm: LLM, news_contents: str) -> str:
    llm.set_prompt(
        f"""
        [요청 사항]
        다음 뉴스 요약들을 종합하여 15개의 문장으로 정리해 주세요.

        [준수 사항]
        각 문장은 ~했어요, ~해요로 종결되는 100자 내외의 완결된 문장이어야 합니다.
        핵심 사건에 대한 기사 문장 5문장, 사건의 배경과 관련된 맥락에 대한 문장 5문장, 사건의 진행과 결과를 설명하는 문장 5문장으로 구성되어야 합니다.


        [참고 사항]
        하나의 뉴스 요약에서, 첫 번째 문장은 이 기사에서 다루는 핵심 사건, 두 번째 문장은 사건의 배경과 관련된 맥락, 세 번째 문장은 사건의 진행과 결과를 설명합니다.
        """
    )
    processed = llm.generate(news_contents)

    llm.set_prompt(
        f"""
        [요청 사항]
        다음 뉴스 초고를 완결된 3문단 구성의 뉴스 기사로 만들어 주세요.

        [준수 사항]
        뉴스 기사는 고등학생들이 읽을 것이에요. 친근한 말투 부탁해요.
        뉴스 기사는 3문단으로 구성되어야 해요.
        뉴스 기사의 모든 문장은 친근한 말투인 ~했어요, ~해요로 종결되어야 해요.
        뉴스 기사의 모든 문장은 100자 내외의 완결된 문장이어야 해요.

        [참고 사항]
        이 초고는 핵심 사건에 대한 기사 문장 5문장, 사건의 배경과 관련된 맥락에 대한 문장 5문장, 사건의 진행과 결과를 설명하는 문장 5문장으로 구성되어 있어요.
        """
    )
    processed = llm.generate(processed)

    llm.set_prompt(
        f"""
        [요청 사항]
        다음 문장들에 대하여, 모든 문장을 ~했어요, ~해요, ~어요로 종결되게 수정해 주세요.

        [예시]
        최근 집콕 트렌드가 확산되면서 홈 인테리어와 관련된 라이프스타일 시장이 급성장하고 있습니다. -> 최근 집콕 트렌드가 확산되면서 홈 인테리어와 관련된 라이프스타일 시장이 급성장하고 있어요.
        """
    )

    result = llm.generate(processed)
    return result

news_contents: list = []
finals: list[Article] = []
for cluster in top_clusters:
    for news_id in clusters[cluster]:
        target_news = get_summerized_news(news_id)
        print(target_news.title)
        news_contents.append(target_news.content)

    result = news_creation_chain('\n\n'.join(i[2:] for i in news_contents))

    finals.append(Article(title=get_summerized_news(clusters[cluster]).title, content=result, news_id=clusters[cluster][:]))

In [ ]:
print(result)

In [ ]:
llm.set_prompt(
    f"""
    [요청 사항]
    다음 뉴스 초고를 완결된 3문단 구성의 뉴스 기사로 만들어 주세요.

    [준수 사항]
    뉴스 기사는 고등학생들이 읽을 것이에요. 친근한 말투 부탁해요.
    뉴스 기사는 3문단으로 구성되어야 해요.
    뉴스 기사의 모든 문장은 친근한 말투인 ~했어요, ~해요로 종결되어야 해요.
    뉴스 기사의 모든 문장은 100자 내외의 완결된 문장이어야 해요.

    [참고 사항]
    이 초고는 핵심 사건에 대한 기사 문장 5문장, 사건의 배경과 관련된 맥락에 대한 문장 5문장, 사건의 진행과 결과를 설명하는 문장 5문장으로 구성되어 있어요.
    """
)

r = llm.generate(result)
print(r)

In [ ]:
llm.set_prompt(
    f"""
    [요청 사항]
    다음 문장들에 대하여, 모든 문장을 ~했어요, ~해요, ~어요로 종결되게 수정해 주세요.

    [예시]
    최근 집콕 트렌드가 확산되면서 홈 인테리어와 관련된 라이프스타일 시장이 급성장하고 있습니다. -> 최근 집콕 트렌드가 확산되면서 홈 인테리어와 관련된 라이프스타일 시장이 급성장하고 있어요.
    """
)

a = llm.generate(r)
print(a)

윤석열 대통령과 명태균 씨의 통화 녹음이 공개되며 여야 간 공방이 격화되었습니다. 이재명 더불어민주당 대표는 대통령 당선자가 공천에 개입한 것 자체가 문제이며, 이를 거짓말로 숨겼다는 점이 더 큰 문제라고 주장했어요. 대통령실은 명태균 씨와의 통화가 취임 전 덕담 수준의 간단한 통화였을 뿐이라고 해명했으나, 야당은 이를 거짓말로 주장해요.

더불어민주당은 윤석열 대통령의 공천 개입 의혹을 시사하는 육성 녹음을 공개하며 강력한 비판을 쏟아냈습니다. 국민의힘은 윤석열 대통령의 녹취 파일이 편집·조작되었다고 주장하며 이를 부인하려 한 민주당을 비판했어요. 윤석열 대통령과 명태균 씨의 통화 녹음이 공개되면서 대통령실의 해명이 야당으로부터 거센 비판을 받고 있어요. 대통령실은 명태균 씨와의 통화가 명확한 해명을 내며 적극 반박했으나, 오히려 의구심을 키우는 결과를 초래했어요.

국회 운영위 국감에서 윤석열 대통령의 공천 개입 의혹과 관련해 여야 간 갈등이 고조되었습니다. 정진석 대통령 비서실장은 윤석열 대통령과 명태균 씨의 통화 내용에 대해 "문제가 없다고 한 것 자체가 더 큰 문제"라고 비판했어요. 윤석열 대통령이 명태균 씨와의 통화로 공천 개입 의혹이 제기되며 공직선거법 적용 여부가 논란이 되었습니다. 더불어민주당은 윤석열 대통령의 국회의원 재보궐선거 공천 개입 논란을 부각시키기 위해 대규모 장외집회를 준비 중이에요.
